In [ ]:
#Bibliotecas para arquivos
import re
from pathlib import Path
import json

#Para leitura e escrita de dados
from collections import defaultdict
import pandas as pd
import numpy as np

#Para Imagem
import matplotlib.pyplot as plt

#Normalização
from sklearn.preprocessing import StandardScaler

#Largura de Banda
from scipy.signal import welch

#Modelos
from sklearn.model_selection import ParameterGrid
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

#Métricas de precisão
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, cohen_kappa_score
from scipy.stats import spearmanr

#Outros
from tqdm.auto import tqdm


In [3]:
#HyperParameters

epoch_time = 2


In [2]:
#Leitura Inicial dos Dados

files_csv = list(Path("GAMEEMO").rglob("*.csv"))
files_names = [p.name for p in files_csv]
files_path = [str(p) for p in files_csv]

#Organizacao dos Dados

# procurar por data['SXX']['raw/pre']['GX']
files = defaultdict(lambda: {
    "pre": {},
    "raw": {}
})

for f in files_path:
    subject = re.search(r"S\d{2}", f).group()
    game = re.search(r"G\d", f).group()

    if "AllRawChannels" in f:
        files[subject]["raw"][game] = f
    else:
        files[subject]["pre"][game] = f

files = dict(files)


In [4]:
df = pd.read_csv(files['S01']['pre']['G1'])
df.head()

,AF3,AF4,F3,F4,F7,F8,FC5,FC6,O1,O2,P7,P8,T7,T8,Unnamed: 14
0,-33.0205,-15.1846,-42.1795,1.6872,42.1793,-1.68720,-5.5436,-3.6154,25.7899,-9.88190,5.5436,7.47180,11.8101,17.1128,NaN
1,-28.6291,-20.0583,-42.5410,-10.4653,35.3100,-15.68600,-19.3110,-2.4344,17.4933,3.24420,18.7081,5.09510,17.3683,3.0708,NaN
2,-21.8497,-10.9006,-32.0346,-2.3656,39.6993,-0.64483,-4.0523,-1.0830,26.8081,-3.45840,8.1861,8.40480,15.1209,9.3940,NaN
3,-25.1185,-10.9702,-32.7641,-3.4287,32.7378,4.69650,-8.6299,-1.7412,16.7637,-9.75860,1.1868,0.91086,4.3315,8.1073,NaN
4,-19.0316,-9.5886,-29.1108,-3.9459,35.3533,0.79929,-12.6914,1.0144,13.1068,-0.73692,8.1054,-1.31300,8.1694,8.3442,NaN


In [4]:
#EDA
subjects = [f"S{i:02d}" for i in range(1, 29)]
games = [f"G{i}" for i in range(1, 5)]
datakinds = ['pre', 'raw']
window = 50

cmap = plt.get_cmap('tab10')
colors = [cmap(i) for i in range(8)]
color_map = {
    ('G1', 'pre'): colors[0],
    ('G1', 'raw'): colors[1],
    ('G2', 'pre'): colors[2],
    ('G2', 'raw'): colors[3],
    ('G3', 'pre'): colors[4],
    ('G3', 'raw'): colors[5],
    ('G4', 'pre'): colors[6],
    ('G4', 'raw'): colors[7],
}


In [7]:
def simplePlot(column, df, subject, game, datakind):

   fig, ax = plt.subplots()
   ax.plot(df.index, df[column])

   
   ax.set(xlabel='time', ylabel='EEG read',
      title=f'{subject}\'s {column} EEG read through time {game} - {datakind}')
   plt.show()

def comparePlot(column, df, subject, game, datakind):

   df_savgol = pd.DataFrame()
   fig, ax = plt.subplots()
   
   line1, = ax.plot(df.index, df[column])

   line1.set_color(color_map[(game,datakind)])
   line1.set_alpha(0.3)
   ax.set(xlabel='time', ylabel='EEG read (with savgol filter)',
      title=f'{subject}\'s {column} EEG read through time {game} - {datakind}')
   
   df_savgol[f'rolling_{column}'] = df[column].rolling(window=window,min_periods=1).mean()

   line2, = ax.plot(df.index, df[f'rolling_{column}'])
   line2.set_color(color_map[(game,datakind)])
   line2.set_alpha(0.9)
      
   plt.show()

def plotEpochBandPower(column, epoch, subject, game, datakind):
   new_epoch = epoch.reset_index()
   
   fig, (ax1,ax2) = plt.subplots(2,1)

   line1, = ax1.plot(new_epoch.index, new_epoch[column])

   line1.set_color(color_map[(game,datakind)])
   line1.set_alpha(0.9)
   ax1.set(xlabel='epoch time', ylabel='Normalized EEG read',
      title=f'{subject}\'s {column} EEG read through time {game} - {datakind}')
   
   signal = new_epoch[column].values
   fs = get_fs()
   freqs, psd = welch(signal, fs)
   line2, = ax2.plot(freqs, psd)

   line2.set_color('red')
   line2.set_alpha(0.9)
   ax2.set(xlabel='frequency', ylabel='Power',
      title=f'{subject}\'s {column} Power Spectral Density of Power Spectrum of the signal {game} - {datakind}')
   
   plt.subplots_adjust(hspace=0.6)
   plt.show()
   
   


In [7]:
'''col = 'AF3'
for game in games:
    for kind in datakinds:
        comparePlot(col,'S01',kind,game)'''

"col = 'AF3'\nfor game in games:\n    for kind in datakinds:\n        comparePlot(col,'S01',kind,game)"

In [ ]:
#Utilização da Frequência

def get_fs():
    df = pd.read_csv(files['S01']['pre']['G1'])
    fs = df.shape[0]//300 +1
    return fs


Limpeza de Dados

In [6]:
#Limpeza

# 1.Re-Referência (CAR)
def CAR(df):
    df_car = df.sub(df.mean(axis=1), axis=0)
    return df_car

# 2. Baseline correction
def base_line(df):
    fs = get_fs()
    baseline_samples = epoch_time * fs
    baseline = df.iloc[:baseline_samples].mean()
    df_base = df - baseline
    return df_base

# 3. Normalização por canal
def normalize(df):
    df_norm = df.copy()
    for col in df_norm.columns:
        scaler = StandardScaler()
        df_norm[col] = scaler.fit_transform(df_norm[col].values.reshape(-1,1))
    return df_norm

# 4. Epoching
def epoching(df):
    fs = get_fs()
    window = epoch_time * fs
    epochs = []
    for start in range(0, len(df) - window, window):
        epoch = df.iloc[start:start+window]
        epochs.append(epoch)
    return epochs

# 5. Remoção de Artefatos
def remove_artifact(epochs):
    clean_epochs = []
    threshold = 3  # depois do z-score
    for epoch in epochs:
        if epoch.abs().max().max() < threshold:
            clean_epochs.append(epoch)
    return clean_epochs

# 6. Criação dos Nomes das Colunas
def getColumnNames(data, files):
    df = pd.read_csv(files['S01']['pre']['G1'])
    df.dropna(axis=1, how='all', inplace=True)
    channels = df.columns.tolist()
    bands = ['delta', 'theta', 'alpha', 'beta', 'gamma']

    features_names = []
    labels_names = list(data['S01']['G1'].keys())

    for ch in channels:
        for band in bands:
            features_names.append(f"{ch}_{band}")
    
    features_names = features_names + ['male','female','age']

    return features_names,labels_names


## Extração de Features

A extração de Features Foi feita da seguinte forma:


### Divisão em faixas de frequências

Primeiramente foi feita a divisão dos sinais em 5 faixas de frequências das ondas [1], que variam de acordo com o estado de alerta do indivíduo.

Delta - 0.5 a 4

Theta - 4 a 8

Alpha - 8 a 13

Beta - 13 a 30

Gamma - Maior que 30

In [ ]:
#Criação de Features

# 1. Largura de Banda
def bandpower(signal, fmin, fmax):
    fs = get_fs()
    freqs, psd = welch(signal, fs)
    idx = np.logical_and(freqs >= fmin, freqs <= fmax)
    
    return np.trapz(psd[idx], freqs[idx])

# 2. Extração de Largura de Banda
def extract_features(epoch):
    features = []

    for col in epoch.columns:
        signal = epoch[col].values

        delta = bandpower(signal, 0.5, 4)
        theta = bandpower(signal, 4, 8)
        alpha = bandpower(signal, 8, 13)
        beta  = bandpower(signal, 13, 30)
        gamma = bandpower(signal, 30, 45)

        features.extend([delta, theta, alpha, beta, gamma])

    return features

# 3. Criação de Features com as Larguras de Banda
def feature_creation(clean_epochs):
    X = []

    for epoch in clean_epochs:
        feat = extract_features(epoch)
        X.append(feat)

    X = np.array(X)
    return X

# 4. Processa um dataframe com as features
def process_dataframe(df, subject, game):
    
    df = CAR(df)
    df = base_line(df)
    df = normalize(df)
    
    epochs = epoching(df)
    clean_epochs = remove_artifact(epochs)
    
    X = []
    tuples = []

    for i, epoch in enumerate(clean_epochs):
        feat = extract_features(epoch)
        tuples.append((subject, game, i+1))            
        X.append(feat)
    return tuples, X

# 5. Extrai as informações do txt

def readGAMEEMOdata():
    with open("gameemodatatxt.txt", "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

# 6. Extrai as labels de um game de um subject

def extract_labels(data, subject, game):
    g = data[subject][game]
    
    return np.array([
        g["satisfied"],
        g["boring"],
        g["horrible"],
        g["calm"],
        g["funny"],
        g["valence"],
        g["arousal"]
    ])

# 7. Encoding do gênero
def encode_gender(g):
    return [0, 1] if g == 1 else [1, 0]

# 8. Normalização da idade
def normalize_age(age, min_age=20, max_age=27):
    return [(age - min_age) / (max_age - min_age)]


# 9. Processa todo o dataset, criando as features novas 
def build_dataset(files):
    
    data = readGAMEEMOdata()
    X = []
    y = []
    
    tuples = []
    
    colsX, colsy = getColumnNames(data, files)
    
    total_files = sum(
        len(files[s]['pre'])
        for s in files
    )

    with tqdm(total=total_files, desc="Total processing") as pbar:
        for subject in subjects:

            # 🔹 dados demográficos
            gender = data[subject]["gender"]
            age = data[subject]["Age"]
            
            gender_feat = encode_gender(gender)
            age_feat = normalize_age(age)
            demo_features = np.array(gender_feat + age_feat)

            for game in games:
                
                pbar.set_postfix({
                    "Subject": subject,
                    "Game": game,
                })

                df = pd.read_csv(files[subject]['pre'][game])
                df.dropna(axis=1, how='all', inplace=True)
                
                idx_tuple, features_epochs = process_dataframe(df, subject, game)
                tuples.append(idx_tuple)
                # 🔹 labels do jogo
                labels = extract_labels(data, subject, game)
                
                for feat in features_epochs:
                    
                    # X
                    feat = np.concatenate([feat, demo_features])
                    X.append(feat)
                    
                    # y (repete para cada epoch)
                    y.append(labels)
                
                pbar.update(1)
    
    index = pd.MultiIndex.from_tuples(
        tuples,
        names=["subject", "game", "epoch"]
    )

    df_features = pd.DataFrame(X, columns=colsX, index=index)
    df_labels = pd.DataFrame(y, columns=colsy, index=index)
    
    return df_features, df_labels




In [8]:
def getIndex(files):
    tuples = []

    total = len(subjects) * len(games)

    with tqdm(total=total, desc="Processing files") as pbar:
        for subject in subjects:
            for game in games:
                
                pbar.set_postfix({
                    "Subject": subject,
                    "Game": game
                })

                df = pd.read_csv(files[subject]['pre'][game])
                df.dropna(axis=1, how='all', inplace=True)
                
                df = CAR(df)
                df = base_line(df)
                df = normalize(df)
                
                epochs = epoching(df)
                clean_epochs = remove_artifact(epochs)
                for i, epoch in enumerate(clean_epochs):
                    tuples.append((subject, game, i+1))
                    
                pbar.update(1)
        index = pd.MultiIndex.from_tuples(
            tuples,
            names=["subject", "game", "epoch"]
        )
                
    return index



In [ ]:
#Leitura de Labels


df_features, df_labels = build_dataset(files)

[0.  1.  0.5]


Total processing:   0%|          | 0/112 [00:00<?, ?it/s, Subject=S01, Game=G1]


ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [11]:
df_features.to_pickle('features_SML.pkl')
df_labels.to_pickle('labels_SML.pkl')

Divisão de Dados

In [ ]:
df_features = pd.read_pickle('features_SML.pkl')
df_labels = pd.read_pickle('labels_SML.pkl')

In [ ]:
def createLOSOSplit(features, labels, test_subject, target, remove_demo = False):
    if remove_demo:
        features.drop(['male','female','age'], axis=1, inplace=True)

    X_test = features.xs(test_subject, level=0)
    y_test = labels.xs(test_subject, level=0)[target]

    X_train = features.drop(test_subject, level=0)
    y_train = labels.drop(test_subject, level=0)[target]

    return np.array(X_train), np.array(y_train), np.array(X_test), np.array(y_test)

def LOSOCV(model, features, labels, metric_list):
    
    dfs = {}

    metrics = {}
    
    for target in tqdm(labels.columns, desc="Targets", leave=False):

        results_subject = {}
        metrics_target = {}

        for test_subject in tqdm(subjects, desc=f"Subject {test_subject} - {target}", leave=False):
            results_subject[test_subject] = {}

            X_train, y_train, X_test, y_test = createLOSOSplit(features, labels, test_subject, target)

            scaler = StandardScaler()
            scaler.fit(X_train)
            X_train = scaler.transform(X_train)
            X_test = scaler.transform(X_test)

            model_fold = clone(model)

            model_fold.fit(X_train, y_train)
            y_pred = np.round(model_fold.predict(X_test))
            
            for metric in metric_list:
                if metric == "mae":
                    results_subject[test_subject]["mae"] = mean_absolute_error(y_test, y_pred)
                if metric == "rmse":
                    results_subject[test_subject]["rmse"] = np.sqrt(mean_squared_error(y_test, y_pred))
                if metric == "r2":
                    results_subject[test_subject]["r2"] = r2_score(y_test, y_pred)
                if metric == "qwk":
                    results_subject[test_subject]["qwk"] = cohen_kappa_score(y_test, np.round(y_pred), weights='quadratic')
                if metric == "spearmanr":
                    results_subject[test_subject]["spearmanr"] = spearmanr(y_test, y_pred)[0]
        
        results_df = pd.DataFrame.from_dict(results_subject, orient='index')
        dfs[target] = results_df
        
        for metric in metric_list:
            metrics_target[metric] = {}
            values = [results_subject[s][metric] for s in results_subject]
            metrics_target[metric]['mean'] = np.mean(values)
            metrics_target[metric]['std'] = np.std(values)
        metrics[target] = metrics_target
    return dfs, metrics


Criação do Modelo

In [ ]:
xgb_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 6]
}

rf_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10]
}

knn_grid = {
    "n_neighbors": [3, 5, 7]
}

grids = {
    "XGB": xgb_grid,
    "RF": rf_grid,
    "KNN": knn_grid
}

base_models = {
    "XGB": XGBRegressor(objective='reg:squarederror', eval_metric = 'mae'),
    "RF": RandomForestRegressor(),
    "KNN": KNeighborsRegressor()
}

metrics_all = {}
results_all = {}

metric_list = ["mae", "rmse", "r2", "qwk", "spearmanr"]
total_grid = sum(len(list(ParameterGrid(grid))) for grid in grids.values())

#XGBOOST
with tqdm(total=total_grid, desc="Total processing") as pbar:
    for model_name in base_models:
        
        base_model = base_models[model_name]

        pbar.set_description(f"modelo atual {model_name}") 

        for params in ParameterGrid(grids[model_name]):

            model = clone(base_model)  # ✅ SEMPRE aqui
            model.set_params(**params)
            
            dfs, metrics = LOSOCV(model, df_features, df_labels, metric_list)
            
            key = f"{model_name}_{params}"
            
            results_all[key] = dfs
            metrics_all[key] = metrics

            pbar.update(1)



Total processing:   0%|          | 0/23 [00:00<?, ?it/s]

Targets:   0%|          | 0/7 [00:00<?, ?it/s]

Subjects (satisfied):   0%|          | 0/28 [00:00<?, ?it/s]

Subjects (boring):   0%|          | 0/28 [00:00<?, ?it/s]

Subjects (horrible):   0%|          | 0/28 [00:00<?, ?it/s]

...

Subjects (boring):   0%|          | 0/28 [00:00<?, ?it/s]

Subjects (horrible):   0%|          | 0/28 [00:00<?, ?it/s]

Subjects (calm):   0%|          | 0/28 [00:00<?, ?it/s]

Subjects (funny):   0%|          | 0/28 [00:00<?, ?it/s]

Subjects (valence):   0%|          | 0/28 [00:00<?, ?it/s]

In [ ]:
with pd.ExcelWriter("LOSOCV_results.xlsx") as writer:
    for target, df in results_all.items():
        df.to_excel(writer, sheet_name=target)

with pd.ExcelWriter("LOSOCV_metric.xlsx") as writer:
    for target, df in metrics.items():
        df.to_excel(writer, sheet_name=target)

y_true classes: [3 4 8 9]
y_pred classes: [2 3 4 5 6 7 8 9]


c:\Users\mathe\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:2480: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


## Fontes

[1] https://www.ncbi.nlm.nih.gov/books/NBK390342/?utm_source=chatgpt.com